# DepthWizard: Monocular Depth & Elevation Estimation Pipeline
This notebook demonstrates the end-to-end transformation of single-view satellite optical imagery into calibrated Digital Surface Models (DSMs) and 3D terrain meshes.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.ingestion import load_image, export_geotiff
from src.depth_estimation import DepthEstimator
from src.calibration import ScaleCalibrator
from src.mesh_generator import MeshGenerator
from src.evaluation import calculate_metrics, landscape_error_analysis, generate_synthetic_reference

## 1. Image Ingestion & Metadata Extraction

In [ ]:
image_path = '../data/input/sample.jpg'
rgb_img, meta = load_image(image_path)
print(f"Loaded image shape: {rgb_img.shape}, Georeferenced: {meta.get('is_georeferenced', False)}")

## 2. Monocular Depth Estimation (Depth Anything V2)

In [ ]:
estimator = DepthEstimator(model_name='vits')
rel_depth, raw_depth = estimator.predict_depth(rgb_img)
print(f"Depth map range: [{rel_depth.min():.3f}, {rel_depth.max():.3f}]")

## 3. Scale Calibration to Metric DSM / rDSM

In [ ]:
calibrator = ScaleCalibrator(srtm_cache_dir='../data/srtm_dem')
calibrated_dsm, a, b, mode = calibrator.calibrate_depth_to_metric(rel_depth, meta=meta)
print(f"Mode: {mode}, Scaling a={a:.2f}, Offset b={b:.2f}")

## 4. Export Standard Geospatial GeoTIFF & 3D Mesh

In [ ]:
export_geotiff(calibrated_dsm, '../data/output/output_dsm.tif', meta=meta)
mesh_gen = MeshGenerator()
mesh_gen.generate_elevation_heatmap(calibrated_dsm, '../data/output/elevation_heatmap.png')
mesh_path = mesh_gen.create_textured_mesh(rgb_img, calibrated_dsm, '../data/output/terrain_model.glb')
print(f"3D Mesh saved to: {mesh_path}")

## 5. Quantitative Accuracy & Landscape Stability Validation

In [ ]:
ref_dsm = generate_synthetic_reference(calibrated_dsm)
metrics = calculate_metrics(calibrated_dsm, ref_dsm)
landscape = landscape_error_analysis(calibrated_dsm, ref_dsm)
print("Validation Metrics:", metrics)
print("Landscape Breakdown:", landscape)